# Annotation baseline notebook
Unexecuted code-only template.

Install the package and obtain authorized inputs separately. No datasets or results are included. Run from the package root; use environment variables in the setup cell to configure local paths. Write executed notebooks and generated files to an ignored local output directory.

## 1. Environment and paths


In [ ]:
from pathlib import Path
import os, sys, json, platform, hashlib, subprocess
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'saa_baseline').is_dir(), 'Run from package root or its notebooks/ folder.'
sys.path.insert(0, str(ROOT / 'src'))
DATA = Path(os.environ.get('SAA_DATA_DIR', ROOT / 'data'))
OUT = Path(os.environ.get('SAA_OUTPUT_DIR', ROOT / 'results'))
PRIVATE = Path(os.environ.get('SAA_PRIVATE_DIR', ROOT / 'private_annotations'))
LEGACY = os.environ.get('SAA_LEGACY_DIR')
OUT.mkdir(parents=True, exist_ok=True)
import numpy as np, pandas as pd, scipy, sklearn
from saa_baseline.data import normalize_annotations, analysis_data, partition_masks, sha256
from saa_baseline.analysis import reproduce
from saa_baseline.notebook_runner import display_table, display_png
raw_path = DATA / 'SPACEDOS_ISS2020_slice.csv'
marks_path = DATA / 'kakona_reference_marks.csv'
print({'Python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
       'scipy': scipy.__version__, 'sklearn': sklearn.__version__})

## 2. Normalize local annotations
Create a separate normalized copy and provenance sidecar without modifying the input. This creates no new human labels.


In [ ]:
normalization = normalize_annotations(marks_path,
    PRIVATE / 'martin_reference_marks_normalized.csv',
    PRIVATE / 'martin_reference_marks_provenance.json')
assert normalization['non_annotator_bytes_preserved'] is True
print({k: normalization[k] for k in ['rows', 'columns_preserved', 'non_annotator_bytes_preserved',
                                    'source_sha256', 'normalized_sha256']})
print('Only annotator is normalized. No independent human review or added labels.')

## 3. Run synthetic software tests
Passing software tests is not scientific validation.


In [ ]:
env = dict(os.environ, PYTHONPATH=str(ROOT / 'src'))
check = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT / 'tests'), '-v'],
                       capture_output=True, text=True, env=env, cwd=ROOT)
(OUT / 'unit_tests.txt').write_text(check.stdout + check.stderr)
print('\n'.join((check.stdout + check.stderr).splitlines()[-5:]))
assert check.returncode == 0, 'Unit tests failed; do not continue.'

## 4. Load fixed inputs and partitions
Input guards reject unsupported versions and invalid boundaries. Inspect the implementation for the fixed input schema and configuration.


In [ ]:
frame, marks, human, cadence, hashes = analysis_data(raw_path, marks_path)
partitions = partition_masks(frame, human)
display_table(pd.DataFrame([{'partition': name, 'rows': int(mask.sum()),
                            'first_observation': frame.loc[mask, 'UTC'].min(),
                            'last_observation': frame.loc[mask, 'UTC'].max()}
                           for name, mask in partitions.items()]))
print({'analysis_rows': len(frame), 'human_intervals': len(human),
       'annotators': 1, 'median_cadence_seconds': cadence,
       'acquisition_gaps_over_30_seconds': int(frame.UTC.diff().dt.total_seconds().gt(30).sum())})

## 5. Run the baseline
Generate local results using the package configuration, without parameter retuning.


In [ ]:
manifest = reproduce(raw_path, marks_path, OUT, legacy_dir=LEGACY)
display_table(pd.read_csv(OUT / 'retrospective_model_agreement.csv'))
print('Frozen configuration:', manifest['parameters'], 'threshold=', manifest['threshold'])
print('Legacy prediction comparison:', manifest['legacy_prediction_parity'])
assert manifest['independent_final_test_used'] is False
assert manifest['publication_authorized'] is False

## 6. Inspect interval-support sensitivity
Support pieces are computational representations, not additional physical events. Compare supported intervals with legacy envelopes without treating either as ground truth.


In [ ]:
display_table(pd.read_csv(OUT / 'support_sensitivity.csv'))
print('Corrected supported-run overlap:')
display_table(pd.read_csv(OUT / 'supported_event_overlap.csv'))
print('Legacy-envelope overlap, kept for transparent comparison:')
display_table(pd.read_csv(OUT / 'legacy_envelope_event_overlap.csv'))
runs = pd.read_csv(OUT / 'model_runs.csv')
print({'predicted_runs': len(runs), 'support_pieces': int(runs.support_pieces.sum()),
       'runs_with_supported_duration_under_60_seconds': int((runs.supported_seconds < 60).sum())})

## 7. Inspect timing and containment sensitivity
These are descriptive sensitivity calculations, not selected clock corrections or evidence of physical truth.


In [ ]:
display_table(pd.read_csv(OUT / 'timing_sensitivity.csv'))
print({'human_events': len(human),
       'fully_contained_in_legacy_SRAG_envelopes': manifest['human_events_fully_contained_legacy'],
       'fully_contained_in_capped_SRAG_support': manifest['human_events_fully_contained_supported']})
print('Containment changes describe support conventions, not new physical truth.')

## 8. Display a local illustration
This cell displays a locally generated figure only after authorized inputs are provided. The figure is not distributed in the repository.


In [ ]:
display_png(OUT / 'continuous_window.png')

## 9. Record local code provenance
No independent-evaluation, publication-permission or new-human-annotation claim follows from running this notebook. The repository MIT license covers included code, not external inputs or generated research artifacts.


In [ ]:
source_manifest = {str(p.relative_to(ROOT)): sha256(p)
                   for p in sorted((ROOT / 'src').rglob('*.py'))}
(OUT / 'code_sha256.json').write_text(json.dumps(source_manifest, indent=2) + '\n')
print('Source files hashed:', len(source_manifest))
print('Completed local retrospective reproduction. No new human annotation or independent-test claim.')